**Questão 1**

Alterar os nomes das colunas

In [1]:
import numpy as np
import pandas as pd
df = pd.read_csv('saude_do_sono_estilo_vida.csv')

In [2]:
df.rename(columns={
    'ID': 'Identificador',
    'Pressão sanguíneaaaa':'Pressão sanguínea',
    'Ocupação': 'Profissão',
    'Categoria BMI': 'IMC'
}, inplace=True)

In [3]:
print(df.columns)

Index(['Identificador', 'Gênero', 'Idade', 'Profissão', 'Duração do sono',
       'Qualidade do sono', 'Nível de atividade física', 'Nível de estresse',
       'IMC', 'Pressão sanguínea', 'Frequência cardíaca', 'Passos diários',
       'Distúrbio do sono'],
      dtype='object')


Questão 2:Qual é a média, a moda e a mediana de horas de sono para cada uma das profissões?[‘mean’, np.median, pd.Series.mode]

In [4]:
# --- Questão 2 Corrigida ---
import numpy as np
import pandas as pd

# Descobre o nome real da coluna de sono para não dar erro
coluna_sono = 'Duração do sono (horas)' if 'Duração do sono (horas)' in df.columns else 'Duração do sono'
coluna_profissao = 'Profissão' if 'Profissão' in df.columns else 'Ocupação'

estatisticas_sono = df.groupby(coluna_profissao)[coluna_sono].agg(
    Média='mean',
    Mediana=np.median,
    Moda=lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
).reset_index()

display(estatisticas_sono)

/tmp/ipykernel_24327/4231249315.py:9: FutureWarning: The provided callable <function median at 0x7b2b7b777d80> is currently using SeriesGroupBy.median. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "median" instead.
  estatisticas_sono = df.groupby(coluna_profissao)[coluna_sono].agg(


,Profissão,Média,Mediana,Moda
0,Advogado(a),7.410638,7.3,7.2
1,Cientista,6.000000,6.0,5.8
2,Contador(a),7.113514,7.2,7.2
3,Enfermeiro(a),7.048611,6.5,6.1
4,Eng. de Software,6.750000,6.8,7.5
5,Engenheiro(a),7.987302,8.3,8.4
6,Gerente,6.900000,6.9,6.9
7,Médico(a),6.970423,7.6,6.0
8,Pessoa Vendendora,6.403125,6.4,6.5
9,Professor(a),6.690000,6.6,6.6


**Questão 3:** Cálculo de Impacto de Obesidade em Eng. Software

Das pessoas que atuam com engenharia de software qual a porcentagem de obesos?

In [5]:
# ==========================================
# CENTRAL DE COORDENAÇÃO - QUESTÃO 3
# ==========================================
import numpy as np
import pandas as pd
import os

# 1. Tenta encontrar o arquivo na pastinha do Colab de qualquer jeito
arquivos_na_pasta = os.listdir('/content') if os.path.exists('/content') else []
arquivo_csv = None

for f in arquivos_na_pasta:
    if 'saude_do_sono' in f and f.endswith('.csv'):
        arquivo_csv = f"/content/{f}"
        break

if arquivo_csv is None:
    # Se não achou na pasta /content, tenta ler direto pelo nome padrão
    arquivo_csv = 'saude_do_sono_estilo_vida.csv'

try:
    # 2. Carrega o arquivo de dados
    df = pd.read_csv(arquivo_csv)

    # 3. Força a padronização dos nomes das colunas (O que a Questão 1 deveria fazer)
    df.rename(columns={
        'ID': 'Identificador',
        'Ocupação': 'Profissão',
        'Categoria BMI': 'IMC',
        'Duração do sono (horas)': 'Duração do sono'
    }, inplace=True, errors='ignore')

    print("✅ Arquivo carregado e colunas sincronizadas com sucesso!\n")

    # 4. EXECUÇÃO DA QUESTÃO 3
    # Filtrando a profissão exata (Engenheiro(a) de software)
    df_eng_software = df[df['Profissão'] == 'Engenheiro(a) de software']
    cont_imc_eng = df_eng_software['IMC'].value_counts()
    total_imc_eng = cont_imc_eng.sum()

    if total_imc_eng > 0:
        profissao_eng = df_eng_software['Profissão'].values[0]
        print(f"Total de {profissao_eng} na base: {total_imc_eng}\n")

        # Criando a tabela resumo com as porcentagens
        df_imc_eng = cont_imc_eng.reset_index(name='Total')
        df_imc_eng['Total %'] = (df_imc_eng['Total'] * 100) / total_imc_eng

        # Mostra a tabelinha na tela
        display(df_imc_eng)

        # Busca a porcentagem de obesos para dar a resposta final
        if 'Obeso' in df_imc_eng['IMC'].values:
            obesos_perc_print = df_imc_eng.loc[df_imc_eng['IMC'] == 'Obeso', 'Total %'].values[0]
            print(f"\n🚀 Resposta Final: {profissao_eng} com Obesidade corresponde a {obesos_perc_print:.1f}%")
        else:
            print(f"\n🚀 Resposta Final: Não existem profissionais de {profissao_eng} classificados com Obesidade nesta base.")
    else:
        print("❌ Erro: Não encontramos a profissão 'Engenheiro(a) de software' na base de dados.")
        print("Verifique se o arquivo CSV correto foi arrastado para a pastinha lateral.")

except Exception as e:
    print("❌ Não foi possível carregar os dados.")
    print(f"Erro do Python: {e}")
    print("\n👉 SOLUÇÃO RÁPIDA: Clique no ícone de Pastinha (📁) do lado esquerdo do Colab e arraste o arquivo do seu computador para lá novamente!")

✅ Arquivo carregado e colunas sincronizadas com sucesso!

❌ Erro: Não encontramos a profissão 'Engenheiro(a) de software' na base de dados.
Verifique se o arquivo CSV correto foi arrastado para a pastinha lateral.


**Questão 4:** Comparativo de Sono: Advocacia ou Vendas vs. média geral

De acordo com os dados, advogar ou ser representante de vendas faz você dormir menos? (Use o método ‘isin’, considere a média)

In [6]:
# Retorna a media geral de sono da base
media_sono = df['Duração do sono'].mean()
media_sono

np.float64(7.129490616621984)

In [7]:
# Filtra os dados de acordo com as profissoes solicitadas
filtro_profissao = df['Profissão'].isin(['Advogado(a)', 'Representante de Vendas'])
df_adv_vendas = df[filtro_profissao]
df_adv_vendas.head()

,Identificador,Gênero,Idade,Profissão,Duração do sono,Qualidade do sono,Nível de atividade física,Nível de estresse,IMC,Pressão sanguíneaaaa,Frequência cardíaca,Passos diários,Distúrbio do sono
3,4,Homem,28,Representante de Vendas,5.9,4,30,8,Obesidade,140/90,85,3000,Apneia do sono
4,5,Homem,28,Representante de Vendas,5.9,4,30,8,Obesidade,140/90,85,3000,Apneia do sono
93,94,Homem,35,Advogado(a),7.4,7,60,5,Obesidade,135/88,84,3300,Apneia do sono
109,110,Homem,37,Advogado(a),7.4,8,60,5,Normal,130/85,68,8000,Nenhuma
111,112,Homem,37,Advogado(a),7.4,8,60,5,Normal,130/85,68,8000,Nenhuma


In [8]:
# Imprime a media geral para comparar
print(f"Média Geral de Sono da Base: {media_sono:.2f} horas")

# Traz a media de cada profissao
media_sono_adv_vendas = df_adv_vendas.groupby('Profissão')['Duração do sono'].agg('mean')
media_sono_adv_vendas

Média Geral de Sono da Base: 7.13 horas


,Duração do sono
Profissão,
Advogado(a),7.410638
Representante de Vendas,5.900000


In [9]:
# A partir daqui, e adicional, porque o que ja foi feito antes responde

# Funcao que compara as medias do grupo com a media geral da base
def comparar_media_sono(medias_grupo, media_geral=media_sono):
  # Percorre cada profissao e sua media no grupo de medias
  for profissao, media_grupo in medias_grupo.items():
    # Checa se a media do grupo e menor que a media geral
    if media_grupo < media_geral:
      return profissao, media_grupo
  return

# Atribui os valores retornados da funcao chamada
profissao_print, media_print = comparar_media_sono(media_sono_adv_vendas)

# Imprime resposta
print(f"Resposta: Ser {profissao_print} faz você dormir menos (Média: {media_print:.2f}h)")


Resposta: Ser Representante de Vendas faz você dormir menos (Média: 5.90h)


**Questão 5**

Entre quem fez enfermagem e quem fez medicina, quem tem menos horas de sono?

In [10]:
# Cria um subconjunto apenas com as profissões de interesse
enf_med = ['Enfermeiro(a)','Médico(a)']
saude = df[df['Profissão'].isin(enf_med)]

# Calcula a média de duração do sono para cada profissão
media = saude.groupby('Profissão')['Duração do sono'].mean().reset_index()

# Ordena o resultado
media = media.sort_values(by='Duração do sono')

# Exibe o resultado para cada profissão
print(media)
print('\n')

# Identifica o profissional com menos hora de sono
menos = media['Profissão'].iloc[0]

# Exibe o resultado
print('O profissional com menos hora de sono é o(a)', menos)

print('\n')

       Profissão  Duração do sono
1      Médico(a)         6.970423
0  Enfermeiro(a)         7.048611


O profissional com menos hora de sono é o(a) Médico(a)




**Questão 6**

Criar subconjuntos com as colunas: Identificador, gênero, idade, pressão sanguínea e frequência cardíaca

In [11]:
# --- Questão 6 Corrigida (À Prova de Falhas) ---

# 1. Identifica o ID (pode ser 'Identificador' ou 'ID')
col_id = 'Identificador' if 'Identificador' in df.columns else 'ID'

# 2. Descobre o nome exato da coluna de Pressão vasculhando os nomes existentes
col_pressao = [col for col in df.columns if 'Press' in col or 'press' in col][0]

# 3. Descobre o nome exato da coluna de Frequência Cardíaca
col_fc = [col for col in df.columns if 'Freq' in col or 'freq' in col][0]

print(f"Colunas detectadas com sucesso:")
print(f"- ID: '{col_id}'")
print(f"- Pressão: '{col_pressao}'")
print(f"- Frequência: '{col_fc}'\n")

# 4. Cria o subconjunto com os nomes reais encontrados
subconjunto = df[[col_id, 'Gênero', 'Idade', col_pressao, col_fc]]

# Mostra o resultado na tela
display(subconjunto.head())

Colunas detectadas com sucesso:
- ID: 'Identificador'
- Pressão: 'Pressão sanguíneaaaa'
- Frequência: 'Frequência cardíaca'



,Identificador,Gênero,Idade,Pressão sanguíneaaaa,Frequência cardíaca
0,1,Homem,27,126/83,77
1,2,Homem,28,125/80,75
2,3,Homem,28,125/80,75
3,4,Homem,28,140/90,85
4,5,Homem,28,140/90,85


In [12]:
print(subconjunto.head())

   Identificador Gênero  Idade Pressão sanguíneaaaa  Frequência cardíaca
0              1  Homem     27               126/83                   77
1              2  Homem     28               125/80                   75
2              3  Homem     28               125/80                   75
3              4  Homem     28               140/90                   85
4              5  Homem     28               140/90                   85


In [13]:
subconjunto.head()

,Identificador,Gênero,Idade,Pressão sanguíneaaaa,Frequência cardíaca
0,1,Homem,27,126/83,77
1,2,Homem,28,125/80,75
2,3,Homem,28,125/80,75
3,4,Homem,28,140/90,85
4,5,Homem,28,140/90,85


**Questão 7**

Descubra qual a profissão menos frequente no conjunto.

In [14]:
# --- Questão 7 Corrigida ---
coluna_profissao = 'Profissão' if 'Profissão' in df.columns else 'Ocupação'

# Calcula a frequência real
df_frequencia = df[coluna_profissao].value_counts(normalize=True) * 100
print(df_frequencia)
print('\n')

# Encontra a profissão menos frequente usando o idxmin()
menos_freq = df[coluna_profissao].value_counts().idxmin()

print('O profissional menos frequente na amostra de entrevistados é o(a):', menos_freq)

Profissão
Enfermeiro(a)              19.302949
Médico(a)                  19.034853
Engenheiro(a)              16.890080
Advogado(a)                12.600536
Professor(a)               10.723861
Contador(a)                 9.919571
Pessoa Vendendora           8.579088
Cientista                   1.072386
Eng. de Software            1.072386
Representante de Vendas     0.536193
Gerente                     0.268097
Name: proportion, dtype: float64


O profissional menos frequente na amostra de entrevistados é o(a): Gerente
